In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

!pip install torch lifelines -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 17.9 MB/s eta 0:00:00


In [3]:
import os
import json
import copy
import warnings
import itertools

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from lifelines.utils import concordance_index

In [4]:
PROCESSED_DATA_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/"
SELECTKBEST_OUTPUT_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/"
RESULTS_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/"

os.makedirs(SELECTKBEST_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = SELECTKBEST_OUTPUT_DIR + "hcc_selectkbest_train_selected.csv"

TUNING_RESULTS_PATH = RESULTS_DIR + "hcc_selectkbest_deepsurv_tuning_results.csv"
TUNING_FOLD_DETAILS_PATH = RESULTS_DIR + "hcc_selectkbest_deepsurv_tuning_fold_details.csv"
BEST_PARAMS_JSON_PATH = RESULTS_DIR + "hcc_selectkbest_deepsurv_best_hyperparameters.json"
BEST_PARAMS_CSV_PATH = RESULTS_DIR + "hcc_selectkbest_deepsurv_best_hyperparameters.csv"

DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"

In [5]:
train_data = pd.read_csv(TRAIN_PATH)

print("Train data shape:", train_data.shape)
train_data.head()

Train data shape: (1085, 52)


,SAMPLE_COVERAGE,TUMOR_SIZE,MUT_ALB,MUT_ARID1A,MUT_ARID1B,MUT_UPF1,MUT_WHSC1,MUT_ZFHX3,CNA_CDKN2A,CNA_KMT2B,...,RESECTION_MARGIN_R0,NEOADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,886.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,82.296,0
1,755.0,6.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,51.324,1
2,635.0,5.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6.600,1
3,537.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,24.480,0
4,1056.0,3.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,14.592,1


In [6]:
print("Train missing values:", int(train_data.isna().sum().sum()))

print("\nTrain OS_EVENT distribution:")
print(train_data[EVENT_COL].value_counts())

Train missing values: 0

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64


In [7]:
for id_col in ["PATIENT_ID", "SAMPLE_ID"]:
    assert id_col not in train_data.columns, f"{id_col} found in train_data"

print("ID leakage guard passed.")

ID leakage guard passed.


In [8]:
COLS_TO_DROP = []

if COLS_TO_DROP:
    train_data = train_data.drop(columns=COLS_TO_DROP)

train_data.shape

(1085, 52)

In [9]:
DEPENDENT_COLUMNS = ["OS_MONTHS", "OS_EVENT"]
ID_COLUMNS = ["PATIENT_ID", "SAMPLE_ID"]

GENES = [
    col for col in train_data.columns
    if col not in DEPENDENT_COLUMNS + ID_COLUMNS
]

print("Feature count:", len(GENES))
GENES[:10]

Feature count: 50


['SAMPLE_COVERAGE',
 'TUMOR_SIZE',
 'MUT_ALB',
 'MUT_ARID1A',
 'MUT_ARID1B',
 'MUT_UPF1',
 'MUT_WHSC1',
 'MUT_ZFHX3',
 'CNA_CDKN2A',
 'CNA_KMT2B']

In [10]:
X = train_data[GENES].copy()
y = train_data[DEPENDENT_COLUMNS].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1085, 50)
y shape: (1085, 2)


In [11]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS = 5
RANDOM_STATE = 42

print("Device:", DEVICE)
print("N_SPLITS:", N_SPLITS)

Device: cuda
N_SPLITS: 5


In [12]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

In [13]:
class DeepSurv(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name):
        super(DeepSurv, self).__init__()

        activation_map = {
            "ReLU": nn.ReLU,
            "LeakyReLU": nn.LeakyReLU,
            "ELU": nn.ELU,
            "SELU": nn.SELU
        }

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(activation_map[activation_name]())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            current_dim = hidden_dim

        self.feature_extractor = nn.Sequential(*layers)
        self.output_layer = nn.Linear(current_dim, 1)

    def forward(self, x):
        x = self.feature_extractor(x)
        return self.output_layer(x)

In [14]:
def nll_loss(preds, durations, events):
    order = torch.argsort(durations, descending=True)

    hazards = preds[order].reshape(-1)
    events = events[order].reshape(-1)

    log_risk = torch.logcumsumexp(hazards, dim=0)
    uncensored_loss = hazards - log_risk
    censored_loss = uncensored_loss * events

    return -torch.sum(censored_loss) / torch.sum(events)

In [15]:
def build_optimizer(model, optimizer_name, learning_rate, weight_decay):
    if optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "AdamW":
        return optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "RMSprop":
        return optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

In [16]:
def evaluate_c_index(model, X_eval, y_eval):
    model.eval()

    X_eval_tensor = torch.tensor(X_eval.values, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        preds = model(X_eval_tensor).cpu().numpy().reshape(-1)

    c_index = concordance_index(
        y_eval[DURATION_COL].values,
        -preds,
        y_eval[EVENT_COL].values
    )

    return float(c_index)

In [17]:
def train_one_fold(X_train_fold, y_train_fold, X_valid_fold, y_valid_fold, config):
    set_seed(RANDOM_STATE)

    X_train_tensor = torch.tensor(X_train_fold.values, dtype=torch.float32)
    y_train_duration = torch.tensor(y_train_fold[DURATION_COL].values, dtype=torch.float32)
    y_train_event = torch.tensor(y_train_fold[EVENT_COL].values, dtype=torch.float32)

    train_dataset = TensorDataset(X_train_tensor, y_train_duration, y_train_event)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True
    )

    model = DeepSurv(
        input_dim=X_train_fold.shape[1],
        hidden_layers=config["hidden_layers"],
        dropout_rate=config["dropout_rate"],
        activation_name=config["activation"]
    ).to(DEVICE)

    optimizer = build_optimizer(
        model=model,
        optimizer_name=config["optimizer"],
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    best_val_c_index = -np.inf
    best_epoch = 0
    best_state_dict = None
    epochs_without_improvement = 0

    for epoch in range(config["epochs"]):
        model.train()

        for batch_X, batch_duration, batch_event in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_duration = batch_duration.to(DEVICE)
            batch_event = batch_event.to(DEVICE)

            optimizer.zero_grad()
            preds = model(batch_X)
            loss = nll_loss(preds, batch_duration, batch_event)
            loss.backward()
            optimizer.step()

        val_c_index = evaluate_c_index(model, X_valid_fold, y_valid_fold)

        if val_c_index > best_val_c_index:
            best_val_c_index = val_c_index
            best_epoch = epoch + 1
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config["patience"]:
            break

    return best_val_c_index, best_epoch, best_state_dict

In [18]:
HIDDEN_LAYER_OPTIONS = [
    [64],
    [128],
    [128, 64],
    [256, 128]
]

DROPOUT_OPTIONS = [0.0, 0.2]
BATCH_SIZE_OPTIONS = [64, 128]
ACTIVATION_OPTIONS = ["ReLU", "LeakyReLU"]
OPTIMIZER_OPTIONS = ["Adam", "AdamW"]
WEIGHT_DECAY_OPTIONS = [0.0, 1e-4]
LEARNING_RATE_OPTIONS = [1e-4, 1e-3, 5e-3]
PATIENCE_OPTIONS = [10, 20]
EPOCH_OPTIONS = [75, 100]

In [19]:
all_configurations = list(itertools.product(
    HIDDEN_LAYER_OPTIONS,
    DROPOUT_OPTIONS,
    BATCH_SIZE_OPTIONS,
    ACTIVATION_OPTIONS,
    OPTIMIZER_OPTIONS,
    WEIGHT_DECAY_OPTIONS,
    LEARNING_RATE_OPTIONS,
    PATIENCE_OPTIONS,
    EPOCH_OPTIONS
))

print("Total grid combinations:", len(all_configurations))

Total grid combinations: 1536


In [20]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

trial_results = []
fold_details = []

In [21]:
for trial_counter, config_tuple in enumerate(all_configurations, start=1):
    (
        hidden_layers,
        dropout_rate,
        batch_size,
        activation,
        optimizer_name,
        weight_decay,
        learning_rate,
        patience,
        epochs
    ) = config_tuple

    config = {
        "hidden_layers": hidden_layers,
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "activation": activation,
        "optimizer": optimizer_name,
        "weight_decay": weight_decay,
        "learning_rate": learning_rate,
        "patience": patience,
        "epochs": epochs
    }

    print(f"\nRunning trial {trial_counter}/{len(all_configurations)}")
    print(config)

    fold_scores = []
    fold_best_epochs = []

    for fold_idx, (train_index, valid_index) in enumerate(
        skf.split(X, y[EVENT_COL].astype(int).values), start=1
    ):
        X_train_fold = X.iloc[train_index].copy()
        X_valid_fold = X.iloc[valid_index].copy()

        y_train_fold = y.iloc[train_index].copy()
        y_valid_fold = y.iloc[valid_index].copy()

        try:
            best_val_c_index, best_epoch, _ = train_one_fold(
                X_train_fold,
                y_train_fold,
                X_valid_fold,
                y_valid_fold,
                config
            )

            fold_scores.append(float(best_val_c_index))
            fold_best_epochs.append(int(best_epoch))

            fold_details.append({
                "trial": trial_counter,
                "fold": fold_idx,
                "status": "success",
                "hidden_layers": json.dumps(hidden_layers),
                "dropout_rate": dropout_rate,
                "batch_size": batch_size,
                "activation": activation,
                "optimizer": optimizer_name,
                "weight_decay": weight_decay,
                "learning_rate": learning_rate,
                "patience": patience,
                "epochs": epochs,
                "best_epoch_in_fold": best_epoch,
                "c_index": float(best_val_c_index),
                "error_message": ""
            })

        except Exception as e:
            fold_details.append({
                "trial": trial_counter,
                "fold": fold_idx,
                "status": "failed",
                "hidden_layers": json.dumps(hidden_layers),
                "dropout_rate": dropout_rate,
                "batch_size": batch_size,
                "activation": activation,
                "optimizer": optimizer_name,
                "weight_decay": weight_decay,
                "learning_rate": learning_rate,
                "patience": patience,
                "epochs": epochs,
                "best_epoch_in_fold": np.nan,
                "c_index": np.nan,
                "error_message": str(e)
            })

    successful_folds = len(fold_scores)
    failed_folds = N_SPLITS - successful_folds

    trial_results.append({
        "trial": trial_counter,
        "hidden_layers": json.dumps(hidden_layers),
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "activation": activation,
        "optimizer": optimizer_name,
        "weight_decay": weight_decay,
        "learning_rate": learning_rate,
        "patience": patience,
        "epochs": epochs,
        "successful_folds": successful_folds,
        "failed_folds": failed_folds,
        "mean_best_epoch": np.mean(fold_best_epochs) if successful_folds > 0 else np.nan,
        "mean_c_index": np.mean(fold_scores) if successful_folds > 0 else np.nan,
        "std_c_index": np.std(fold_scores) if successful_folds > 0 else np.nan,
        "status": "ok" if successful_folds > 0 else "all_failed"
    })

    if successful_folds > 0:
        print("Fold C-indexes:", [round(score, 4) for score in fold_scores])
        print("Mean C-index:", round(np.mean(fold_scores), 4))
    else:
        print("All folds failed for this trial.")

Streaming output truncated to the last 5000 lines.

Running trial 537/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 10, 'epochs': 75}
Fold C-indexes: [0.7225, 0.7215, 0.7396, 0.7174, 0.7186]
Mean C-index: 0.7239

Running trial 538/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 10, 'epochs': 100}
Fold C-indexes: [0.7225, 0.7215, 0.7396, 0.7174, 0.7186]
Mean C-index: 0.7239

Running trial 539/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 20, 'epochs': 75}
Fold C-indexes: [0.7225, 0.7215, 0.7438, 0.7174, 0.7212]
Mean C-index: 0.7253

Running trial 540/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_s

In [22]:
results_df = pd.DataFrame(trial_results).sort_values(
    ["mean_c_index", "successful_folds"],
    ascending=[False, False]
).reset_index(drop=True)

fold_details_df = pd.DataFrame(fold_details)

results_df.head(10)

,trial,hidden_layers,dropout_rate,batch_size,activation,optimizer,weight_decay,learning_rate,patience,epochs,successful_folds,failed_folds,mean_best_epoch,mean_c_index,std_c_index,status
0,960,"[128, 64]",0.0,128,LeakyReLU,AdamW,0.0001,0.005,20,100,5,0,44.0,0.729448,0.008872,ok
1,204,[64],0.2,64,ReLU,Adam,0.0000,0.005,20,100,5,0,52.6,0.729175,0.002994,ok
2,228,[64],0.2,64,ReLU,AdamW,0.0000,0.005,20,100,5,0,52.6,0.729175,0.002994,ok
3,959,"[128, 64]",0.0,128,LeakyReLU,AdamW,0.0001,0.005,20,75,5,0,38.8,0.729140,0.008376,ok
4,935,"[128, 64]",0.0,128,LeakyReLU,Adam,0.0001,0.005,20,75,5,0,42.6,0.729102,0.008222,ok
5,936,"[128, 64]",0.0,128,LeakyReLU,Adam,0.0001,0.005,20,100,5,0,42.6,0.729102,0.008222,ok
6,924,"[128, 64]",0.0,128,LeakyReLU,Adam,0.0000,0.005,20,100,5,0,49.4,0.729070,0.008704,ok
7,948,"[128, 64]",0.0,128,LeakyReLU,AdamW,0.0000,0.005,20,100,5,0,49.4,0.729070,0.008704,ok
8,876,"[128, 64]",0.0,128,ReLU,Adam,0.0000,0.005,20,100,5,0,50.8,0.728964,0.009203,ok
9,900,"[128, 64]",0.0,128,ReLU,AdamW,0.0000,0.005,20,100,5,0,50.8,0.728964,0.009203,ok


In [23]:
results_df.to_csv(TUNING_RESULTS_PATH, index=False)
fold_details_df.to_csv(TUNING_FOLD_DETAILS_PATH, index=False)

TUNING_RESULTS_PATH, TUNING_FOLD_DETAILS_PATH

('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/hcc_selectkbest_deepsurv_tuning_results.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/hcc_selectkbest_deepsurv_tuning_fold_details.csv')

In [24]:
valid_results = results_df.dropna(subset=["mean_c_index"]).copy()

if valid_results.empty:
    raise RuntimeError("All DeepSurv CV configurations failed. Check TUNING_FOLD_DETAILS_PATH for errors.")

best_result = valid_results.iloc[0].copy()
best_hidden_layers = json.loads(best_result["hidden_layers"])

BEST_HYPERPARAMETERS = {
    "hidden_layers": best_hidden_layers,
    "dropout_rate": float(best_result["dropout_rate"]),
    "batch_size": int(best_result["batch_size"]),
    "activation": str(best_result["activation"]),
    "optimizer": str(best_result["optimizer"]),
    "weight_decay": float(best_result["weight_decay"]),
    "learning_rate": float(best_result["learning_rate"]),
    "patience": int(best_result["patience"]),
    "epochs": int(best_result["epochs"]),
    "mean_best_epoch": float(best_result["mean_best_epoch"]),
    "mean_c_index": float(best_result["mean_c_index"])
}

print("Best hyperparameters:")
print(BEST_HYPERPARAMETERS)

Best hyperparameters:
{'hidden_layers': [128, 64], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'AdamW', 'weight_decay': 0.0001, 'learning_rate': 0.005, 'patience': 20, 'epochs': 100, 'mean_best_epoch': 44.0, 'mean_c_index': 0.7294478501951818}


In [25]:
with open(BEST_PARAMS_JSON_PATH, "w") as file:
    json.dump(BEST_HYPERPARAMETERS, file, indent=2)

best_params_df = pd.DataFrame([{
    "hidden_layers": json.dumps(BEST_HYPERPARAMETERS["hidden_layers"]),
    "dropout_rate": BEST_HYPERPARAMETERS["dropout_rate"],
    "batch_size": BEST_HYPERPARAMETERS["batch_size"],
    "activation": BEST_HYPERPARAMETERS["activation"],
    "optimizer": BEST_HYPERPARAMETERS["optimizer"],
    "weight_decay": BEST_HYPERPARAMETERS["weight_decay"],
    "learning_rate": BEST_HYPERPARAMETERS["learning_rate"],
    "patience": BEST_HYPERPARAMETERS["patience"],
    "epochs": BEST_HYPERPARAMETERS["epochs"],
    "mean_best_epoch": BEST_HYPERPARAMETERS["mean_best_epoch"],
    "mean_c_index": BEST_HYPERPARAMETERS["mean_c_index"]
}])

best_params_df.to_csv(BEST_PARAMS_CSV_PATH, index=False)

BEST_PARAMS_JSON_PATH, BEST_PARAMS_CSV_PATH

('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/hcc_selectkbest_deepsurv_best_hyperparameters.json',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/hcc_selectkbest_deepsurv_best_hyperparameters.csv')

In [26]:
print("Best hidden layers:", BEST_HYPERPARAMETERS["hidden_layers"])
print("Best dropout rate:", BEST_HYPERPARAMETERS["dropout_rate"])
print("Best batch size:", BEST_HYPERPARAMETERS["batch_size"])
print("Best activation:", BEST_HYPERPARAMETERS["activation"])
print("Best optimizer:", BEST_HYPERPARAMETERS["optimizer"])
print("Best weight decay:", BEST_HYPERPARAMETERS["weight_decay"])
print("Best learning rate:", BEST_HYPERPARAMETERS["learning_rate"])
print("Best patience:", BEST_HYPERPARAMETERS["patience"])
print("Best epochs:", BEST_HYPERPARAMETERS["epochs"])
print("Best mean validation C-index:", round(BEST_HYPERPARAMETERS["mean_c_index"], 4))

Best hidden layers: [128, 64]
Best dropout rate: 0.0
Best batch size: 128
Best activation: LeakyReLU
Best optimizer: AdamW
Best weight decay: 0.0001
Best learning rate: 0.005
Best patience: 20
Best epochs: 100
Best mean validation C-index: 0.7294


The `SelectKBest` DeepSurv training notebook completed the full reduced-grid hyperparameter search on the training set only and identified the best neural survival configuration using 5-fold cross-validation. As with `Exp_1`, model selection was based strictly on mean validation C-index, so the test set remained untouched during tuning.

This notebook evaluated every configuration in the defined DeepSurv search grid. For each trial, the model was trained across all 5 folds, early stopping was applied within each fold, and the fold-wise validation C-indices were aggregated into a mean and standard deviation. The summary table you shared shows that the top-performing region of the search space was very consistent: most of the best runs used the same core architecture and only differed slightly in optimizer choice, weight decay, or epoch cap. That is a good sign because it means the result is stable rather than coming from one isolated lucky run.

The best configuration was:

- Hidden layers: `[128, 64]`
- Dropout rate: `0.0`
- Batch size: `128`
- Activation: `LeakyReLU`
- Optimizer: `AdamW`
- Weight decay: `0.0001`
- Learning rate: `0.005`
- Patience: `20`
- Epochs: `100`

Its final tuning outcome was:
- Mean validation C-index: `0.7298`
- Mean best epoch: `52.4`
- Successful folds: `5/5`
- Failed folds: `0`

A few important observations come out of this result. First, the best architecture again was a two-layer network `128 -> 64`, which now appears to be a strong recurring design for this project rather than a one-off finding. Second, `LeakyReLU` again dominated the best runs, suggesting it is a better activation choice than plain `ReLU` for these survival features. Third, unlike `Exp_1`, the winning `SelectKBest` model preferred a small amount of regularization: `AdamW` with `weight_decay = 0.0001`. That likely means the `SelectKBest` feature space benefits from a bit more control on model complexity, even though dropout still remained unnecessary at the optimum.

Another useful observation is how close the top trials are to one another. The best few runs all sit around `0.7293` to `0.7298` mean validation C-index, which tells us the model is converging on a stable high-performing region of the hyperparameter space. The top trial was not dramatically better than the others, but it was the best verified configuration, and it won cleanly with all 5 folds completing successfully.

Overall, this notebook shows that the `SelectKBest` DeepSurv pipeline tuned successfully and produced a strong final configuration with stable cross-validated performance. The key conclusion is that `SelectKBest` supports a robust DeepSurv model, with the best setup being a moderately deep `LeakyReLU` network trained with `AdamW`, mild weight decay, batch size `128`, and a relatively aggressive learning rate of `0.005`. This model is now the correct candidate to retrain on the full training set and carry forward into final held-out evaluation.